In [ ]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import polars.selectors as cs
import pandas as pd
import plotly.express as px

import ipaddress
import datetime as dt

import polars.selectors as cs

In [ ]:
df_path = r'/mnt/samsung/Datasets/spotify_2026_scenarios.csv'

In [ ]:
df = pl.read_csv(df_path)

In [ ]:
df.collect_schema()

In [ ]:
df.estimated_size('mb')

In [ ]:
df = df.with_columns(
    pl.col('duration_ms').cast(pl.Duration('ms')).alias('duration_ms'),
)

In [ ]:
numeric_cols = df.select(cs.numeric()).columns

df = df.with_columns([
    df[col].shrink_dtype() for col in numeric_cols
])

In [ ]:
df.collect_schema()

In [ ]:
df.null_count()

In [ ]:
# df.filter(
#     pl.col('track_name').is_null()
# )

null = df.null_count().unpivot(variable_name='column', value_name='nulls')
null.filter(
    pl.col('nulls') > 0
)

In [ ]:
df.select(
    'popularity', 'tempo', 'loudness', 'duration_ms'
).describe().filter(
    pl.col('statistic').str.contains('min') |
    pl.col('statistic').str.contains('max')
)

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.select(
    pl.col('artist_name').n_unique(),
    pl.col('scenario').n_unique(),
).unpivot()

In [ ]:
df['scenario'].value_counts()

In [ ]:
df['ai_generated'].value_counts()

In [ ]:
df['short_form'].value_counts()

In [ ]:
for col in ['scenario', 'ai_generated', 'short_form']:
    print(f'\n--- {col} ---\n')
    print(df[col].value_counts())

In [ ]:
df.group_by(
    'ai_generated'
).agg(
    pl.len().alias('count')
).with_columns(
    ((pl.col('count') / pl.col('count').sum()) * 100).round(2).alias('percent')
).sort(
    'ai_generated'
)

In [ ]:
df = df.with_columns(
    pl.col('duration_ms').dt.total_seconds().alias('duration_seconds'),
    pl.col('duration_ms').dt.total_minutes().alias('duration_minutes'),
)

In [ ]:
df.select(
    pl.col('duration_minutes').mean().alias('duration_minutes_mean'),
    pl.col('duration_minutes').median().alias('duration_minutes_median'),
    pl.col('duration_minutes').max().alias('duration_minutes_max'),
).unpivot()

In [ ]:
df.filter(
    pl.col('duration_ms') < pl.duration(seconds=30)
).height

In [ ]:
df.select(
    pl.col('key').count().alias('key_count'),
    pl.col('key').n_unique().alias('key_unique'),
)

In [ ]:
df['key'].unique()

In [ ]:
encoded_key = {
    0: 'C',
    1: 'C#',
    2: 'D',
    3: 'D#',
    4: 'E',
    5: 'F',
    6: 'F#',
    7: 'G',
    8: 'G#',
    9: 'A',
    10: 'A#',
    11: 'B',
    -1: 'No Key Detected'
}

In [ ]:
key_encoded_column = df.select(
    pl.col('key').replace_strict(encoded_key, return_dtype=pl.String).alias('key_encoded')
).to_series()

df.insert_column(
    index=9, column=key_encoded_column,
)

In [ ]:
{col for col in enumerate(df.columns)}

In [ ]:
df.select(
    pl.col('key_encoded')
).group_by(
    pl.col('key_encoded')
).len().sort(
    'len', descending=True
)

In [ ]:
df.group_by('mode').len().sort('mode')

In [ ]:
df.with_columns(
    pl.col('mode').replace_strict({0: 'minor', 1: 'major'}, return_dtype=pl.String).alias('mode_label')
).group_by('mode_label').len().sort('len', descending=True)

In [ ]:
df.select(
    pl.col('artist_name'),
    pl.col('popularity'),
).group_by(
    pl.col('artist_name')
).agg(
    pl.len().alias('track_count'),
    pl.mean('popularity').round(2).alias('avg_pop'),
    pl.n_unique('artist_name').alias('unique_scenarios'),
).sort(
    'track_count', descending=True
).head(20)

In [ ]:
features = ['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'speechiness', 'valence']
#
# for col in features:
#     fig = px.histogram(df, x=col, title=f'Distribution of {col}', template='plotly_dark')
#     fig.show(fig.show())

In [ ]:
df_long = df.select(features).unpivot()

fig = px.histogram(
    df_long,
    x='value',
    facet_col='variable',
    facet_col_wrap=3,
    title='Distribution of Features',
    template='plotly_dark',
)

fig.show()

In [ ]:
fig = px.scatter(
    df.sample(5000),
    x='loudness',
    y='energy',
    opacity=0.2,
    title='Loudness vs. Energy',
    labels={'loudness': 'Loudness (dB)', 'energy': 'Energy'},
    template='plotly_dark',
)

fig.show()

In [ ]:
correlation = df.select(
    pl.corr('loudness', 'energy')
).item()

print(f"Pearson correlation between Loudness and Energy: {correlation:.4f}")

quiet_but_energetic = df.filter(
    (pl.col('loudness') < -20) & (pl.col('energy') > 0.7)
)

print(f"Found {quiet_but_energetic.height} quiet but high-energy songs.")

quiet_but_energetic.select('track_name', 'artist_name', 'loudness', 'energy').head()

In [ ]:
fig = px.scatter(
    quiet_but_energetic,
    x='loudness',
    y='energy',
    opacity=0.2,
    title='Loudness vs. Energy',
    labels={'loudness': 'Loudness (dB)', 'energy': 'Energy'},
    template='plotly_dark',
)

fig.show()

In [ ]:
tempo_category = df.select(
    pl.when(pl.col('tempo') <= 90)
    .then(pl.lit('Slow'))
    .when(pl.col('tempo').is_between(90, 140, closed='right'))
    .then(pl.lit('Medium'))
    .otherwise(pl.lit('Fast')).alias('tempo_category')
).to_series()

df.insert_column(13, tempo_category)

In [ ]:
count_tempo_category = df['tempo_category'].value_counts()

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=df,
    x='tempo',
    fill=True,
    color='blue',
    alpha=0.5
)

plt.axvline(90, color='red', linestyle='--')
plt.axvline(140, color='red', linestyle='--')
plt.title("Tempo Distribution with Category Boundaries")
plt.show()

In [ ]:
instrumentalness_percentage = df.select(
    (pl.col('instrumentalness') > 0.5).mean() * 100
).item()

print(f"Percentage of instrumental tracks: {instrumentalness_percentage:.2f}%")

In [ ]:
df.select(
    'track_id', 'track_name', 'instrumentalness', 'tempo', 'energy'
).filter(
    pl.col('instrumentalness') > 0.5
).select(
    pl.mean('tempo').round(2).alias('tempo_mean'),
    pl.mean('energy').round(2).alias('energy_mean'),
)

In [ ]:
df.with_columns(
    is_instrumental = pl.col('instrumentalness') > 0.5
).group_by(
    'is_instrumental'
).agg(
    pl.mean('popularity').round(2).alias('popularity_mean'),
    pl.mean('tempo').round(2).alias('tempo_mean'),
    pl.mean('energy').round(2).alias('energy_mean'),
    pl.len().alias('track_count'),
)

In [ ]:
df.select(
    'artist_name', 'speechiness'
).group_by(
    'artist_name',
).mean().sort(
    'speechiness', descending=True
).head(10)

In [ ]:
df.with_columns(
    speech_type=pl.when(pl.col('speechiness') > 0.5)
    .then(pl.lit('High'))
    .otherwise(pl.lit('Low'))
)


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['speechiness'], bins=20, kde=True, color='blue', alpha=0.5)

plt.xscale('log')
plt.title('Distribution of Speechiness')
plt.xlabel('Speechiness')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df.select(
    'ai_generated'
)

In [ ]:
features = ['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'speechiness', 'valence']

comparison_df = df.group_by(
    'ai_generated'
).agg([
    pl.col(f).mean().alias(f) for f in features
]).sort('ai_generated')
comparison_df

In [ ]:
comparison_df.unpivot(
    index='ai_generated',
    on=features,
    variable_name='feature',
    value_name='mean_value'
)

In [ ]:
summary = comparison_df.transpose(include_header=True, header_name='feature', column_names=['Human', 'AI'])
summary = summary.filter(pl.col('feature') != 'ai_generated')
summary = summary.with_columns(
    diff = (pl.col('AI') - pl.col('Human')).abs()
).sort('diff', descending=True)

print("Features by difference magnitude:")
print(summary)

In [ ]:
df.select(
    pl.col('scenario').unique()
)

In [ ]:
scenario_comparison = (
    df.group_by("scenario")
    .agg(
        track_count = pl.len(),
        popularity_mean = pl.col("popularity").mean().round(2),
        # Proportion is the mean of 0/1 columns
        prop_ai = pl.col("ai_generated").mean().round(4),
        prop_short_form = pl.col("short_form").mean().round(4),
        # Audio Profile Means
        danceability = pl.col("danceability").mean(),
        energy = pl.col("energy").mean(),
        valence = pl.col("valence").mean()
    )
    .with_columns(
        # 3. Create a composite "mainstream" score
        mainstream_score = (pl.col("danceability") + pl.col("energy") + pl.col("valence"))
    )
    .sort("mainstream_score", descending=True)
)

scenario_comparison

In [ ]:
plot_data = scenario_comparison.select("scenario", "danceability", "energy", "valence").to_pandas().set_index("scenario")

sns.heatmap(plot_data, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Scenario Comparison Heatmap")
plt.show()